In [1]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas: 3.0.2  polars: 1.39.3


In [2]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- wsv_pegel ---
WSV_PEGEL_RESPONSE_JSON = '[{"datum":"2020-01-01","wert":5.0}]'
WSV_PEGEL_EMPTY_RESPONSE_JSON = '[]' 

print("✅ Fixtures loaded")


✅ Fixtures loaded


In [3]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_wsv_pegel(response):
    df = pd.read_json(response)
    df = pl.from_pandas(df).lazy()
    return df

In [4]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_wsv_pegel(response):

    df = pl.read_json(response).lazy()
    return df

In [5]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [6]:

# === Tests: wsv_pegel ===
import tempfile

def _wsv_path(payload):
    tmp = tempfile.TemporaryDirectory()
    path = Path(tmp.name) / "response.json"
    path.write_text(payload, encoding="utf-8")
    return tmp, str(path)

try:
    _tmp, _path = _wsv_path(WSV_PEGEL_RESPONSE_JSON)
    try: _r = gen_wsv_pegel(_path); _r.collect()
    finally: _tmp.cleanup()
    print("✅ L1 smoke gen_wsv_pegel: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_wsv_pegel: {type(_e).__name__}: {_e}")

try:
    _tmp, _path = _wsv_path(WSV_PEGEL_RESPONSE_JSON)
    try: _rb = before_wsv_pegel(_path); _rb.collect()
    finally: _tmp.cleanup()
    print("✅ L1 smoke before_wsv_pegel: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_wsv_pegel: {type(_e).__name__}: {_e}")

for _layer, _payload in [("L2 equivalence", WSV_PEGEL_RESPONSE_JSON), ("L3 edge", WSV_PEGEL_EMPTY_RESPONSE_JSON)]:
    try:
        _tmp, _path = _wsv_path(_payload)
        try: _rb = before_wsv_pegel(_path); _rg = gen_wsv_pegel(_path)
        finally: _tmp.cleanup()
        compare(_rb, _rg, f"{_layer} wsv_pegel JSON", check_row_order=True)
    except Exception as _e:
        print(f"❌ {_layer} wsv_pegel JSON: {type(_e).__name__}: {_e}")


✅ L1 smoke gen_wsv_pegel: OK, type= LazyFrame
✅ L1 smoke before_wsv_pegel: OK
✅ L2 equivalence L2 equivalence wsv_pegel JSON: MATCH
✅ L3 edge wsv_pegel JSON: MATCH
